In [1]:
import scipy.io as sio
import numpy as np
import os
import glob
from matplotlib import pyplot as plt
import re
import random
import json
import pandas as pd

In [3]:
def save_data(save_path, filename, tf):
    if os.path.exists(os.path.join(save_path, filename)):
        return
    else:
        sio.savemat(os.path.join(save_path, filename), tf)


In [21]:
def draw_total_data(show_data, files, result_path, filename):
    plt.figure(figsize=(16, 2 * len(files)))
    plt.imshow(show_data, aspect='auto', cmap='seismic', vmin=-3, vmax=3)
    # plt.imshow(show_data, aspect='auto', cmap='seismic')
    plt.colorbar()
    plt.axis('off')  # Turn axis on to show filenames
    plt.savefig(os.path.join(result_path, f'{filename}.png'), bbox_inches='tight', dpi=300)
    plt.close()

In [6]:
data_path = "/path/to/mae/data/Batter"
files = os.listdir(data_path)

lco_path = [os.path.join(data_path, f) for f in files if "LCO#13" in f]


In [ ]:
for dir in lco_path:
    print(dir)
    bname = os.path.basename(dir)
    files = os.listdir(dir)
    files = [f for f in files if f.endswith('.mat')]
    files.sort()
    print(files)
    total_data = []
    result_path = os.path.join("/path/to/mae/vision_batter")
    if not os.path.exists(result_path):
        os.makedirs(result_path)

    for file in files:
        data = sio.loadmat(os.path.join(dir, file))
        tf = data['a'][:, : , 12:20]
        # Generate a 4*128 by 8*128 matrix
        rows = tf.shape[0]
        chans = tf.shape[-1]
        cat_data = tf[:, :, 0]
        for i in range(chans-1):
            cat_data = np.concatenate([cat_data, np.zeros((rows,2)) + 1, tf[:, :, i+1]], axis=1)

        total_data.append(cat_data)
    show_data = total_data[0]
    cols = show_data.shape[1]
    for i in range(len(total_data)-1):
        show_data = np.concatenate([show_data, np.zeros((2,cols)) + 1, total_data[i+1]], axis=0)
    draw_total_data(show_data, files, result_path, f"{bname}_combined")


/path/to/mae/data/Batter/LCO#13_charge_1A
['wo_0_#3D.mat', 'wo_1_#3D.mat', 'wo_2_#3D.mat', 'wo_3_#3D.mat', 'wo_4_#3D.mat', 'wo_5_#3D.mat', 'wo_6_#3D.mat', 'wo_7_#3D.mat']
/path/to/mae/data/Batter/LCO#13_discharge_0.5A
['wo_10_#3D.mat', 'wo_11_#3D.mat', 'wo_12_#3D.mat', 'wo_13_#3D.mat', 'wo_14_#3D.mat', 'wo_15_#3D.mat']
/path/to/mae/data/Batter/LCO#13_discharge_1.5A
['wo_10_#3D.mat', 'wo_11_#3D.mat', 'wo_12_#3D.mat', 'wo_13_#3D.mat', 'wo_14_#3D.mat', 'wo_15_#3D.mat', 'wo_16_#3D.mat']
/path/to/mae/data/Batter/LCO#13_charge_2A
['wo_0_#3D.mat', 'wo_1_#3D.mat', 'wo_2_#3D.mat', 'wo_3_#3D.mat', 'wo_4_#3D.mat', 'wo_5_#3D.mat']
/path/to/mae/data/Batter/LCO#13_charge_1.5A
['wo_0_#3D.mat', 'wo_1_#3D.mat', 'wo_2_#3D.mat', 'wo_3_#3D.mat', 'wo_4_#3D.mat']
/path/to/mae/data/Batter/LCO#13_discharge_2A
['wo_10_#3D.mat', 'wo_11_#3D.mat', 'wo_12_#3D.mat', 'wo_13_#3D.mat', 'wo_14_#3D.mat', 'wo_15_#3D.mat']
/path/to/mae/data/Batter/LCO#13_charge_0.5A
['wo_0_#3D.mat', 'wo_1_#3D.mat', 'wo_2_#3D.mat', 'wo_3_#

In [23]:
save_path = "/path/to/mae/data/lco_unknown/temp_data"
for dir in lco_path:
    print(dir)
    bname = os.path.basename(dir)
    files = os.listdir(dir)
    files = [f for f in files if f.endswith('.mat')]
    files.sort()
    ah_df_f = glob.glob(os.path.join(dir, '*.xlsx'))
    ah_df = pd.read_excel(ah_df_f[0])[["step" , "容量(Ah)"]]
    ah_dict = dict(zip(ah_df["step"], ah_df["容量(Ah)"]))
    
    for file in files:
        stage = re.match(r"wo_(\d+)", file).group(0)
        data = sio.loadmat(os.path.join(dir, file))
        tf = data['a'][:, :, 13:19]
        ah = ah_dict[stage]
        
        mat_data = {
            'data': tf,
            'ah': ah
        }
        
        sio.savemat(os.path.join(save_path, f"{bname}_{stage}.mat"), mat_data)
        
        
        
        


/path/to/mae/data/Batter/LCO#13_charge_1A
/path/to/mae/data/Batter/LCO#13_discharge_0.5A
/path/to/mae/data/Batter/LCO#13_discharge_1.5A
/path/to/mae/data/Batter/LCO#13_charge_2A
/path/to/mae/data/Batter/LCO#13_charge_1.5A
/path/to/mae/data/Batter/LCO#13_discharge_2A
/path/to/mae/data/Batter/LCO#13_charge_0.5A
/path/to/mae/data/Batter/LCO#13_discharge_1A


In [24]:
da = sio.loadmat("/path/to/mae/data/lco_unknown/select_data/LCO#13_charge_0.5A_wo_1.mat")